# 02 — Trial Downsampling and Train/Test Split

## Purpose
Subsample and balance the raw LFP trial arrays produced by notebook 01,
then create a reproducible stratified train/test split.

## Input files (from notebook 01)
- `lfp_trials_{session_id}_probeC.npy`   — shape (N_trials, N_channels, N_samples)
- `stim_labels_{session_id}_probeC.npy`  — shape (N_trials,)

## Output files
- `lfp_train_trials.npy`  — shape (N_train, 65, 624)
- `lfp_train_labels.npy`  — shape (N_train,)
- `lfp_test_trials.npy`   — shape (N_test,  65, 624)
- `lfp_test_labels.npy`   — shape (N_test,)

## What this notebook does
1. Loads raw trials from multiple sessions into memory
2. Filters out `invalid_presentation` trials
3. Subsamples to a balanced target of ~8000 total trials (stratified by stimulus)
4. Crops the spatial/temporal dimensions to (65 channels, 624 samples) to avoid NaN border effects
5. Saves the four split arrays with a fixed random seed (seed=0) for reproducibility

## ⚠️ Note on paths
Update `data_dir` in the cells below to point to your local data directory.

In [1]:
import numpy as np
import pandas as pd

In [5]:
# trials766 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_766640955_probeC.npy')
# stim766 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_766640955_probeC.npy', allow_pickle=True)
# trials767 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_767871931_probeC.npy')
# stim767 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_767871931_probeC.npy', allow_pickle=True)
# trials768 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_768515987_probeC.npy')
# stim768 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_768515987_probeC.npy', allow_pickle=True)
# trials7711 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_771160300_probeC.npy')
# stim7711 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_771160300_probeC.npy', allow_pickle=True)
# trials7719 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_771990200_probeC.npy')
# stim7719 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_771990200_probeC.npy', allow_pickle=True)
# trials774 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_774875821_probeC.npy')
# stim774 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_774875821_probeC.npy', allow_pickle=True)
# trials7782 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_778240327_probeC.npy')
# stim7782 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_778240327_probeC.npy', allow_pickle=True)
# trials7789 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_778998620_probeC.npy')
# stim7789 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_778998620_probeC.npy', allow_pickle=True)
# trials781 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_781842082_probeC.npy')
# stim781 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_781842082_probeC.npy', allow_pickle=True)
# # trials793 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_793224716_probeC.npy')
# # stim793 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_793224716_probeC.npy', allow_pickle=True)
# trials7798 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_779839471_probeC.npy')
# stim7798 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_779839471_probeC.npy', allow_pickle=True)
# # trials821 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_821695405_probeC.npy')
# # stim821 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_821695405_probeC.npy', allow_pickle=True)
# # trials847 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_847657808_probeC.npy')
# # stim847 = np.load('/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_847657808_probeC.npy', allow_pickle=True)

In [13]:
session_ids = [
    766640955, 767871931, 768515987, 
    771160300, 771990200, 774875821,
    778240327, 778998620, 781842082,
    779839471
]

stim_trials_list = []
stim_stims_list = []

for session_id in session_ids:
    trials = np.load(
        f'/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_{session_id}_probeC.npy',
        mmap_mode='r'   # avoid loading entire array into RAM
    )
    stims = np.load(
        f'/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_{session_id}_probeC.npy',
        allow_pickle=True
    )
    stim_trials_list.append(trials)
    stim_stims_list.append(stims)

# Spontaneous sessions (same IDs here, adjust if needed)
sessions_ok = [
    766640955, 767871931, 768515987, 
    771160300, 771990200, 774875821,
    778240327, 778998620, 781842082,
    779839471 
]

spont_trials_list = []
spont_stims_list = []

for session_id in sessions_ok:
    spont_trials = np.load(
        f'/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_{session_id}_spontaneous.npy',
        mmap_mode='r'
    )
    spont_stims = np.load(
        f'/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_{session_id}_spontaneous.npy',
        allow_pickle=True
    )
    spont_trials_list.append(spont_trials)
    spont_stims_list.append(spont_stims)

########################################
# 2) Downsample to get final_trials / labels
########################################

min_trials_per_stim = 1000
target_total = 8000  # adjust if needed

# Flatten labels into a single array
all_stims = np.concatenate(stim_stims_list + spont_stims_list)
session_lengths = [len(arr) for arr in stim_trials_list + spont_trials_list]

# Build mapping of which session and which offset each trial belongs to
session_offsets = np.cumsum([0] + session_lengths[:-1])

trial_session_map = []
trial_local_idx_map = []
for i, offset in enumerate(session_offsets):
    trial_session_map.extend([i] * session_lengths[i])
    trial_local_idx_map.extend(list(range(session_lengths[i])))

# Filter out invalids
mask_valid = (all_stims != 'invalid_presentation')
df = pd.DataFrame({
    'session': np.array(trial_session_map)[mask_valid],
    'local_idx': np.array(trial_local_idx_map)[mask_valid],
    'stimulus': all_stims[mask_valid]
})

guaranteed_indices = []
remaining_indices = []

for stim, grp in df.groupby('stimulus'):
    if len(grp) >= min_trials_per_stim:
        guaranteed = grp.sample(n=min_trials_per_stim, random_state=17)
        remaining = grp.drop(guaranteed.index)
        guaranteed_indices.extend(guaranteed.index.tolist())
        remaining_indices.extend(remaining.index.tolist())
    else:
        guaranteed_indices.extend(grp.index.tolist())

n_to_sample_total = target_total - len(guaranteed_indices)
if n_to_sample_total > 0 and len(remaining_indices) > 0:
    extra_indices = np.random.RandomState(17).choice(
        remaining_indices, size=n_to_sample_total, replace=False
    )
    final_indices = guaranteed_indices + list(extra_indices)
else:
    final_indices = guaranteed_indices

# Extract final sessions / indices / labels
final_sessions = df.iloc[final_indices]['session'].values
final_local_idxs = df.iloc[final_indices]['local_idx'].values
final_stims = df.iloc[final_indices]['stimulus'].values

# Build final_trials (batch-gather per session), shape (N, 65, 624)
final_trials = []
final_labels = []


In [4]:
# sessions_ok = [
#     766640955, 767871931, 768515987, 
#     771160300, 771990200, 774875821,
#     778240327, 778998620, 781842082,
#     779839471 
# ]
# #793224716,
# #847657808
# spont_trials_list = []
# spont_stims_list = []

# for session_id in sessions_ok:
#     spont_trials = np.load(f'/projectnb/cs523aw/students/pangelos/RetreivedData/lfp_trials_{session_id}_spontaneous.npy')
#     spont_stims = np.load(f'/projectnb/cs523aw/students/pangelos/RetreivedData/stim_labels_{session_id}_spontaneous.npy', allow_pickle=True)
#     spont_trials_list.append(spont_trials)
#     spont_stims_list.append(spont_stims)

In [14]:
for trial_idx, (sess_idx, local_idx, stim) in enumerate(
    zip(final_sessions, final_local_idxs, final_stims)
):
    if sess_idx < len(stim_trials_list):
        arr = stim_trials_list[sess_idx][:, :65, :624]
    else:
        arr = spont_trials_list[sess_idx - len(stim_trials_list)][:, :65, :624]

    trial_data = arr[local_idx]

    if np.isnan(trial_data).any():
        print(f"Skipping NaN trial: trial_idx={trial_idx}, sess_idx={sess_idx}, local_idx={local_idx}")
        continue

    final_trials.append(trial_data)
    final_labels.append(stim)

final_trials = np.stack(final_trials)   # (N, 65, 624)
final_stims = np.array(final_labels)    # keep naming consistent below

print("Final shapes after skipping NaNs:", final_trials.shape, final_stims.shape)

########################################
# 4) Stratified train/test split and save
########################################

train_indices = []
test_indices = []
rng = np.random.default_rng(seed=17)

unique_classes = np.unique(final_stims)
for cls in unique_classes:
    cls_indices = np.where(final_stims == cls)[0]
    n_total = len(cls_indices)
    n_train = int(np.round(n_total * 0.7))
    shuffled = rng.permutation(cls_indices)
    train_indices.extend(shuffled[:n_train])
    test_indices.extend(shuffled[n_train:])

# Ensure proper NumPy integer arrays before indexing
final_trials = np.asarray(final_trials)
final_stims = np.asarray(final_stims)

train_indices = np.asarray(train_indices, dtype=int)
test_indices = np.asarray(test_indices, dtype=int)

X_train = final_trials[train_indices]
y_train = final_stims[train_indices]
X_test = final_trials[test_indices]
y_test = final_stims[test_indices]

np.save('lfp_train_trials.npy', X_train)
np.save('lfp_train_labels.npy', y_train)
np.save('lfp_test_trials.npy', X_test)
np.save('lfp_test_labels.npy', y_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Train label counts:\n", pd.Series(y_train).value_counts())
print("Test label counts:\n", pd.Series(y_test).value_counts())

Skipping NaN trial: trial_idx=7213, sess_idx=14, local_idx=56
Skipping NaN trial: trial_idx=7379, sess_idx=14, local_idx=538
Skipping NaN trial: trial_idx=7383, sess_idx=14, local_idx=3
Skipping NaN trial: trial_idx=7495, sess_idx=14, local_idx=37
Skipping NaN trial: trial_idx=7691, sess_idx=14, local_idx=928
Skipping NaN trial: trial_idx=7824, sess_idx=14, local_idx=885
Final shapes after skipping NaNs: (7994, 65, 624) (7994,)
Train: (5596, 65, 624), Test: (2398, 65, 624)
Train label counts:
 dot_motion                        700
drifting_gratings_75_repeats      700
drifting_gratings_contrast        700
flashes                           700
gabors                            700
natural_movie_one_more_repeats    700
natural_movie_one_shuffled        700
spontaneous                       696
dtype: int64
Test label counts:
 dot_motion                        300
drifting_gratings_75_repeats      300
drifting_gratings_contrast        300
flashes                           300
gabors      

In [5]:
# Check regular trials sessions (e.g., first loaded session)
print("Checking first regular trial session...")
arr = trials766           # or pick any other loaded trials variable
print("Shape:", arr.shape)
print("Total NaNs:", np.isnan(arr).sum())
print("Per-sample NaNs:", np.isnan(arr).sum(axis=(1,2)))  # adjust as needed

# Check a single trial
print("First trial regular, any NaN:", np.isnan(arr[0]).any())

# Check spontaneous trials
print("Checking first spontaneous trials session...")
spont_arr = spont_trials_list[0]
print("Shape:", spont_arr.shape)
print("Total NaNs:", np.isnan(spont_arr).sum())
print("First trial spontaneous, any NaN:", np.isnan(spont_arr[0]).any())


Checking first regular trial session...
Shape: (77436, 82, 624)
Total NaNs: 0
Per-sample NaNs: [0 0 0 ... 0 0 0]
First trial regular, any NaN: False
Checking first spontaneous trials session...
Shape: (4669, 82, 624)
Total NaNs: 0
First trial spontaneous, any NaN: False


In [9]:
stim_trials_list = [
    trials766, trials767, trials768, trials7711, trials7719,
    trials774, trials7782, trials7789, trials781, trials793, trials847
]

stim_stims_list = [
    stim766, stim767, stim768, stim7711, stim7719,
    stim774, stim7782, stim7789, stim781, stim793, stim847
]


In [16]:
min_trials_per_stim = 100
target_total = 800

#Flatten labels into a single array, and keep array:session offsets
all_stims = np.concatenate(stim_stims_list + spont_stims_list)
session_lengths = [len(arr) for arr in stim_trials_list + spont_trials_list]

#Build mapping of which session (and which offset in that session) each trial/label belongs to
session_offsets = np.cumsum([0] + session_lengths[:-1])

trial_session_map = []
trial_local_idx_map = []
for i, offset in enumerate(session_offsets):
    trial_session_map.extend([i] * session_lengths[i])
    trial_local_idx_map.extend(list(range(session_lengths[i])))

#Filter out invalids
mask_valid = (all_stims != 'invalid_presentation')
df = pd.DataFrame({
    'session': np.array(trial_session_map)[mask_valid],
    'local_idx': np.array(trial_local_idx_map)[mask_valid],
    'stimulus': all_stims[mask_valid]
})

guaranteed_indices = []
remaining_indices = []

for stim, grp in df.groupby('stimulus'):
    if len(grp) >= min_trials_per_stim:
        guaranteed = grp.sample(n=min_trials_per_stim, random_state=17)
        remaining = grp.drop(guaranteed.index)
        guaranteed_indices.extend(guaranteed.index.tolist())
        remaining_indices.extend(remaining.index.tolist())
    else:
        guaranteed_indices.extend(grp.index.tolist())

n_to_sample_total = target_total - len(guaranteed_indices)
if n_to_sample_total > 0 and len(remaining_indices) > 0:
    extra_indices = np.random.RandomState(17).choice(
        remaining_indices, size=n_to_sample_total, replace=False)
    final_indices = guaranteed_indices + list(extra_indices)
else:
    final_indices = guaranteed_indices

#Extract only the trials you need for final dataset (memory safe!)
final_sessions = df.iloc[final_indices]['session'].values
final_local_idxs = df.iloc[final_indices]['local_idx'].values
final_stims = df.iloc[final_indices]['stimulus'].values

#Build final_trials (batch-gather per session), shape (7000, 65, 624)
final_trials = []
final_labels = []
for trial_idx, (sess_idx, local_idx, stim) in enumerate(zip(final_sessions, final_local_idxs, final_stims)):
    if sess_idx < len(stim_trials_list):
        arr = stim_trials_list[sess_idx][:, :65, :624]
    else:
        arr = spont_trials_list[sess_idx - len(stim_trials_list)][:, :65, :624]

    trial_data = arr[local_idx]
    if np.isnan(trial_data).any():
        print(f"Skipping NaN trial: trial_idx={trial_idx}, sess_idx={sess_idx}, local_idx={local_idx}")
        continue

    final_trials.append(trial_data)
    final_labels.append(stim)

# Convert to arrays after loop
final_trials = np.stack(final_trials)                 # shape: (valid_trials, 65, 624)
final_labels = np.array(final_labels)                 # shape: (valid_trials,)
print("Final shapes after skipping NaNs:", final_trials.shape, final_labels.shape)


Final shapes after skipping NaNs: (80, 65, 624) (80,)


In [20]:
train_indices = []
test_indices = []
rng = np.random.default_rng(seed=17)

unique_classes = np.unique(final_stims)
for cls in unique_classes:
    cls_indices = np.where(final_stims == cls)[0]
    n_total = len(cls_indices)
    n_train = int(np.round(n_total * 0.7))
    # Randomly shuffle the indices for this class
    shuffled = rng.permutation(cls_indices)
    # Assign to train/test
    train_indices.extend(shuffled[:n_train])
    test_indices.extend(shuffled[n_train:])

train_indices = np.array(train_indices)
test_indices = np.array(test_indices)

X_train = final_trials[train_indices]
y_train = final_stims[train_indices]
X_test = final_trials[test_indices]
y_test = final_stims[test_indices]

np.save('lfp_train_trials.npy', X_train)
np.save('lfp_train_labels.npy', y_train)
np.save('lfp_test_trials.npy', X_test)
np.save('lfp_test_labels.npy', y_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Train label counts:\n", pd.Series(y_train).value_counts())
print("Test label counts:\n", pd.Series(y_test).value_counts())


Train: (56, 65, 624), Test: (24, 65, 624)
Train label counts:
 dot_motion                        7
drifting_gratings_75_repeats      7
drifting_gratings_contrast        7
flashes                           7
gabors                            7
natural_movie_one_more_repeats    7
natural_movie_one_shuffled        7
spontaneous                       7
dtype: int64
Test label counts:
 dot_motion                        3
drifting_gratings_75_repeats      3
drifting_gratings_contrast        3
flashes                           3
gabors                            3
natural_movie_one_more_repeats    3
natural_movie_one_shuffled        3
spontaneous                       3
dtype: int64


In [21]:
y_train = np.load('lfp_train_labels.npy', allow_pickle = True)
x_train = np.load('lfp_train_trials.npy').astype('float32')

In [22]:
print("Has NaN in X_train:", np.isnan(X_train).any())
print("Has Inf in X_train:", np.isinf(X_train).any())
print("Unique labels y_train:", np.unique(y_train))
print("Total NaNs in X_train:", np.isnan(X_train).sum())

Has NaN in X_train: False
Has Inf in X_train: False
Unique labels y_train: ['dot_motion' 'drifting_gratings_75_repeats' 'drifting_gratings_contrast'
 'flashes' 'gabors' 'natural_movie_one_more_repeats'
 'natural_movie_one_shuffled' 'spontaneous']
Total NaNs in X_train: 0


In [23]:
y_train = np.load('lfp_train_labels.npy', allow_pickle=True)
y_test = np.load('lfp_test_labels.npy', allow_pickle=True)

print("Unique train labels:", np.unique(y_train, return_counts=True))
print("Unique test labels:", np.unique(y_test, return_counts=True))

# After encoding
print("Encoded train:", np.unique(y_train, return_counts=True))
print("Encoded test:", np.unique(y_test, return_counts=True))


Unique train labels: (array(['dot_motion', 'drifting_gratings_75_repeats',
       'drifting_gratings_contrast', 'flashes', 'gabors',
       'natural_movie_one_more_repeats', 'natural_movie_one_shuffled',
       'spontaneous'], dtype=object), array([7, 7, 7, 7, 7, 7, 7, 7]))
Unique test labels: (array(['dot_motion', 'drifting_gratings_75_repeats',
       'drifting_gratings_contrast', 'flashes', 'gabors',
       'natural_movie_one_more_repeats', 'natural_movie_one_shuffled',
       'spontaneous'], dtype=object), array([3, 3, 3, 3, 3, 3, 3, 3]))
Encoded train: (array(['dot_motion', 'drifting_gratings_75_repeats',
       'drifting_gratings_contrast', 'flashes', 'gabors',
       'natural_movie_one_more_repeats', 'natural_movie_one_shuffled',
       'spontaneous'], dtype=object), array([7, 7, 7, 7, 7, 7, 7, 7]))
Encoded test: (array(['dot_motion', 'drifting_gratings_75_repeats',
       'drifting_gratings_contrast', 'flashes', 'gabors',
       'natural_movie_one_more_repeats', 'natural_movie_

In [8]:
# import matplotlib.pyplot as plt

# stim_types = [
#     'dot_motion', 'drifting_gratings_75_repeats', 'drifting_gratings_contrast',
#     'flashes', 'gabors',
#     'natural_movie_one_more_repeats', 'natural_movie_one_shuffled',
#     'spontaneous'
# ]

# plt.figure(figsize=(14, 12))

# for i, stim in enumerate(stim_types):
#     idx = np.where(stim793 == stim)[0]
#     if len(idx) == 0:
#         continue  # skip missing stim types

#     # Option 1: Plot first trial, channel 0
#     signal = trials793[idx[0], 0, :]
    
#     # Option 2: Plot mean across all trials of this type (optional, uncomment if preferred)
#     # signal = trials793[idx, 0, :].mean(axis=0)
    
#     plt.subplot(3, 3, i + 1)
#     plt.plot(signal)
#     plt.title(stim)
#     plt.xlabel("Samples")
#     plt.ylabel("LFP (V)")

# plt.tight_layout()
# plt.suptitle("Channel 0 LFP Trace by Stimulus Type", y=1.02)
# plt.show()



In [ ]:
# num_spontaneous = np.sum(stim793 == 'spontaneous')
# print(f"Number of 'spontaneous' trials in this session: {num_spontaneous}")
# stim_counts = pd.Series(stim793).value_counts()
# print(stim_counts)

In [ ]:
# min_channels = min(arr.shape[1] for arr in [
#     trials766, trials767, trials768, trials7711, trials7719, trials774,
#     trials7782, trials7789, trials781, trials793, trials847
# ])
# min_samples = min(arr.shape[2] for arr in [
#     trials766, trials767, trials768, trials7711, trials7719, trials774,
#     trials7782, trials7789, trials781, trials793, trials847
# ])

# #For easy looping
# trial_arrays = [trials766, trials767, trials768, trials7711, trials7719, trials774,
#                 trials7782, trials7789, trials781, trials793, trials847]
# stim_arrays = [stim766, stim767, stim768, stim7711, stim7719, stim774,
#                stim7782, stim7789, stim781, stim793, stim847]

# subsampled_trials = []
# subsampled_stims = []

# min_trials_per_stim = 500

# for trials, stims in zip(trial_arrays, stim_arrays):
#     #Align shapes
#     aligned_trials = trials[:, :min_channels, :min_samples]
#     df = pd.DataFrame({'trial_index': np.arange(len(stims)), 'stimulus': stims})
    
#     #Stratified sampling
#     for stim, grp in df.groupby('stimulus'):
#         n_to_sample = min(min_trials_per_stim, len(grp))  #cap if not enough
#         sampled_indices = grp.sample(n=n_to_sample, random_state=17)['trial_index'].values
#         subsampled_trials.append(aligned_trials[sampled_indices])
#         subsampled_stims.append(stims[sampled_indices])

# #Concatenate downsized arrays
# all_trials = np.concatenate(subsampled_trials)
# all_stims = np.concatenate(subsampled_stims)

# print(all_trials.shape)
# print(all_stims.shape)


In [ ]:
# unique_stimuli = np.unique(all_stims)
# n_classes = len(unique_stimuli)
# print(f"Number of stimuli: {n_classes}")

In [ ]:
# df = pd.DataFrame({'trial_index': np.arange(len(all_stims)), 'stimulus': all_stims})
# min_trials_per_stim = 250

# #Always sample at least 250 per stim
# guaranteed_indices = []
# remaining_indices = []

# for stim, grp in df.groupby('stimulus'):
#     if len(grp) >= min_trials_per_stim:
#         # =First, randomly select 250
#         guaranteed = grp.sample(n=min_trials_per_stim, random_state=17)
#         remaining = grp.drop(guaranteed.index)
#         guaranteed_indices.extend(guaranteed['trial_index'].tolist())
#         remaining_indices.extend(remaining['trial_index'].tolist())
#     else:
#         #If less than 250, take all
#         guaranteed_indices.extend(grp['trial_index'].tolist())

# #Calculate how many more to sample to reach ~7k
# n_to_sample_total = 7000 - len(guaranteed_indices)
# if n_to_sample_total > 0:
#     extra_indices = np.random.RandomState(17).choice(
#         remaining_indices, size=n_to_sample_total, replace=False)
#     final_indices = guaranteed_indices + extra_indices.tolist()
# else:
#     final_indices = guaranteed_indices  #If already >= 7000


In [ ]:
# final_trials = all_trials[final_indices]
# final_stims = all_stims[final_indices]
# print(final_trials.shape, final_stims.shape)
# stim_counts = pd.Series(final_stims).value_counts()
# print("Number of trials per stimulus in final set:\n", stim_counts)

In [ ]:
# mask = (final_stims != 'invalid_presentation')
# final_trials = final_trials[mask]
# final_stims = final_stims[mask]


In [ ]:
# final_trials = all_trials[final_indices]
# final_stims = all_stims[final_indices]
# print(final_trials.shape, final_stims.shape)
# stim_counts = pd.Series(final_stims).value_counts()
# print("Number of trials per stimulus in final set:\n", stim_counts)